# Project 2: Pairwise global alignment with general and affine gap cost

**Group 10**

Emma Réka Erdei

Martina Chabadová

Juan Nicolás Quintero Quintero

---

## Introduction
For this project we have implemented **global_general**, that calculates a global algnment of two strings using **general gap cost** and a **backtracking algorithm**. Our other function **global_affine** finds also a global alignment, but with the constraint of **affine gapcost**, however in exchange for this it takes at most quadratic time. Everything works as expected, and our results match the provided test cases.

## Methods

To implement our solution, we use three different files, one as a helper and two containing the main functions to align the sequences:

### Helper Functions (`alignment_utils.py`)
* `read_fasta`: Reads sequences from FASTA files using the `Bio.SeqIO` library.
* `read_cost_matrix`: Reads text file with the cost matrix and the gapcost formated as following:
    n g
    where n is the size of the alphabet and g is the linear gapcost

    next n lines:
    letter cost1 cost2 ... costn
    
    Where the letter is the i'th letter of the alphabet and each row defines the cost of a substitution
    of alphabet[i] to the rest of the alphabet
* `check_sequences_validity`: Checks the validity of all the sequences, verifies that they don't have characters outside of our alphabet   
* `write_alignment_in_fasta`: Writes the output alignment in a fasta file.

### Global General Alignment (`global_general.py`)

Function to get the global pairwise alignment using general gap cost
    It receives as inputs:
    
        - sequences: a list of two sequences that we want to align
        - alphabet: a list of the current alphabet, aligned with the index of the cost matrix
        - cost_matrix: a matrix with the scores of each substitution
        - gap_cost: a function that will be called to calculate the gapcost
        - gapcost_params: a list with the parameters [a,b] for the gapcost function, for the affine gapcost it has the form a+b*k
        - backtracking: a boolean variable to return an optimal alignment using backtracking, if desired 

We implemented an algorithm to find the global pairwise alignment using general gap cost using an iterative Dynamic Programming (DP) approach, where we fill the alignemt cost matrix iteratively.
* **Initialization**: We create a DP table of dimensions $(n+1) \times (m+1)$. The first row and column are initialized with gap costs according to the gapcost function parametrized by the relevant index of the cell, and the parameters of the gapcost function ($\text{gap\_cost(i, gapcost\_params)}$), this reduces the number of iterations and comparisons. This comes from the observation that the alignments in the first row and in the first column can only come from a succession of insertions or deletions, and that at the nth index we are using the score of a gap of n length.
* **Implementation**: The table is filled iteratively (top-left to bottom-right). In each step the algorithm chooses the option with the minimum cost, with the potential options being a match/mismatch (same as in the linear gapcost case), or a gap of some length (gapcost function application).
* **Backtracking**: If requested, a separate function traverses the score matrix from $(n, m)$ back to $(0, 0)$ to reconstruct the optimal alignment. We used an iterative approach this time that looks first for replacements and then evaluates gaps going up or to the left in the score matrix.

### Global Affine Alignment (`global_affine.py`)

We implemented an affine global optimal cost finding algorithm using an iterative Dynamic Programming (DP) approach. 
It receives as inputs:
        - sequences: a list of two sequences that we want to align
        - alphabet: a list of the current alphabet, aligned with the index of the cost matrix
        - cost_matrix: a matrix with the scores of each substitution
        - a: gap opening penalty
        - b: gap extension penalty
        - backtracking: a boolean variable to return an optimal alignment using backtracking, if desired 
* **Initialization**: We create matricies S (score matrix), D (deletion matrix), and I (insertion matrix) of dimensions $(n+1) \times (m+1)$. We first initialize them with *inf*, then we set the first element of S to 0, and the first row and column are initialized with gap costs scaled by their index in S and in I or D respectively.
* **Implementation**: The table is filled iteratively (top-left to bottom-right) using the helper matricies I and D. The final cost of the optimal global alignment can be found in the S table at $(n, m)$.
* **Backtracking**: If requested, a separate function generally traverses the score matrix from $(n, m)$ back to $(0, 0)$. The backtracking also relies on I and D beside S, as it reconstructs the optimal alignment.

## Tests


In order to verify the correctness of our implementation, we first tested it with short simple sequences, for basic validity, and also utilized the fact that the affine gapcost is a special case of the general gap cost, therefore we could compare their outputs for identical inputs as well.

After verifying correctness in this way, we moved onto the examples provided for the project, testing them for both functions.

For the test cases we verified both the alignment and its score, also utilizing the fact that the *global_general* function can be tested with given affine gap costs as well.

Finally, we took the questions provided and used our implementation to answer them:

### Question 1 (`question1.py`)

Compute the optimal score of an optimal alignment for each pair of the
5 sequences above using program global_affine with the score matrix M 
and gap cost g(k)=5+5*k using your program global_affine. The result
is a 5x5 table where entry (i,j) the optimal score of an alignment of 
seq_i and seq_j.

*Result:*

  ```
    0  266  242  243  256
  266    0  283  259  254
  242  283    0  269  243
  243  259  269    0  247
  256  254  243  247    0
  ```


### Question 2 (`question2.py`)

Compute the optimal score of an optimal alignment for each pair of the
5 sequences above using program global_general with the score matrix M 
and gap cost g(k)=5+5*k using your program global_affine. The result
is a 5x5 table where entry (i,j) the optimal score of an alignment of 
seq_i and seq_j.

*Result for the global_general:*

```
Global general function -> O(n^3)
    0  266  242  243  256
  266    0  283  259  254
  242  283    0  269  243
  243  259  269    0  247
  256  254  243  247    0
It took 1.6297 seconds.
```

*Result for the global_affine:*

```
Global general function -> O(n^3)
    0  266  242  243  256
  266    0  283  259  254
  242  283    0  269  243
  243  259  269    0  247
  256  254  243  247    0
It took 1.6297 seconds.
```

### Question 3 (`question3.py`)

Compute and show an optimal alignment of sequence 1 and 2 using gap 
cost g(k)=5+5*k using your programs general_affine and general_general.

*Result for the global_general:*

```
atggagagaataaaagaactgagagatct-aatgtcgcagtcccgcac-tcgcgagatactcactaagac-cactgtggaccatatggccataatcaaaaag
-atggatgtcaatccgactctacttttcctaaaaattccagcgcaaaatgccataagcaccacattcccttatactggagatcctcca--tacagccatggaa
```

*Result for the global_affine:*

```
tatggagagaataaaagaactgagagatct-aatgtcgcagtcccgcac-tcgcgagatactcactaagac-cactgtggaccatatggccataatcaaaaag
-atggatgtcaatccgactctacttttcctaaaaattccagcgcaaaatgccataagcaccacattcccttatactggagatcctcca--tacagccatggaa
```


## Experiments

To verify that our implementation matches the theoretical running time of our solution (O(n*m)), we generated multiple random sequences of variable size, and computed the optimal cost of an alignment between them, registering the running time of filling the score matrix. 

To facilitate the analysis, we decided to make the alignment assuming both sequences are of the same length, thus, out solution should have a running time of O(n**2)

In [ ]:
import pickle
import matplotlib.pyplot as plt
import pandas as pd

file = open("tests/result.pkl",'rb')
results = pickle.load(file)
file.close()

results = pd.DataFrame(results)
results = results[results["size"] >10]

plt.scatter(results["size"], results["time"])
plt.xlabel("n")
plt.ylabel("T(n)")
plt.title("Running time of the implementation")
plt.show()

plt.scatter(results["size"], results["time"]/(results["size"]**2))
plt.xlabel("n")
plt.ylabel("T(n)/n^2")
plt.title("Running time of the implementation vs theoretical (O(n**2))")
plt.show()

As we can see, the running time of the implementation is O(n**2), showing a horizontal line when compared with the theoretical time in the second plot with come deviations when the input size is around 600, this is most likely a hardware deviation, generated by other processes running at the same time as the program. 

Thus, we can conclude that the implementation is correct.